# Phase 9 — Model Evaluation and Comparison

**Goal:** compare the four Phase 8 regression models on identical walk-forward folds and against the persistence baseline.

**Protected test:** January–February 2017 is not used for model selection.

## 1. Load Phase 8 and Phase 7 evidence

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_config import *
from src.modeling_utils import *

import json
import numpy as np
import pandas as pd

phase8_results = pd.read_csv(PHASE8_RESULTS) if PHASE8_RESULTS.exists() else None
baseline_path = AUDIT_DIR / "phase7_walk_forward_persistence_baseline.csv"
baseline = pd.read_csv(baseline_path) if baseline_path.exists() else None

if phase8_results is None:
    print("Phase 8 results not found. Run 08_model_training.ipynb with the model flags enabled.")
else:
    display(phase8_results.head())

if baseline is not None:
    display(baseline.head())
else:
    print("Phase 7 persistence baseline not found. The model comparison can still run, but baseline comparison will be unavailable.")

,model,fold,train_rows,validation_rows,train_MAE,validation_MAE,train_RMSE,validation_RMSE,train_R2,validation_R2
0,Linear Regression,fold_1,82553,87927,12.796282,13.176943,22.644095,25.225431,0.904739,0.897477
1,Linear Regression,fold_2,170480,87088,12.775927,13.541397,23.393264,29.213670,0.905808,0.879936
2,Linear Regression,fold_3,257568,86826,13.716414,13.116795,25.204244,24.655488,0.898341,0.894447
3,Decision Tree,fold_1,82553,87927,7.477353,14.985260,12.978002,26.719557,0.968709,0.884972
4,Decision Tree,fold_2,170480,87088,7.643777,14.163271,13.070443,28.937505,0.970596,0.882196


,fold,MAE,RMSE,R2
0,fold_1,10.425537,18.995444,0.941864
1,fold_2,10.395576,19.501142,0.946499
2,fold_3,9.578905,18.419303,0.941090
3,Mean,10.133339,18.971963,0.943151


## 2. Aggregate validation performance

In [2]:
REQUIRED_MODELS = {
    "Linear Regression",
    "Decision Tree",
    "Random Forest",
    "HistGradientBoosting",
}

comparison = None

if phase8_results is not None and not phase8_results.empty:
    available_models = set(phase8_results["model"].unique())
    missing_models = REQUIRED_MODELS - available_models

    if missing_models:
        print("Phase 8 is incomplete. Missing models:", sorted(missing_models))
        print("Run the corresponding model blocks in 08_model_training.ipynb.")
    else:
        rows = []
        for model_name, group in phase8_results.groupby("model"):
            rows.append({
                "model": model_name,
                "mean_validation_MAE": group["validation_MAE"].mean(),
                "mean_validation_RMSE": group["validation_RMSE"].mean(),
                "mean_validation_R2": group["validation_R2"].mean(),
                "validation_RMSE_std": group["validation_RMSE"].std(ddof=0),
                "mean_train_RMSE": group["train_RMSE"].mean(),
                "RMSE_train_to_validation_gap": (
                    group["validation_RMSE"].mean() - group["train_RMSE"].mean()
                ),
            })

        comparison = pd.DataFrame(rows)

        if baseline is not None and "RMSE" in baseline.columns:
            baseline_rows = baseline.copy()
            if "fold" in baseline_rows.columns:
                baseline_rows = baseline_rows[
                    baseline_rows["fold"].astype(str).str.lower() != "mean"
                ]
            if not baseline_rows.empty:
                baseline_rmse = baseline_rows["RMSE"].mean()
                comparison["baseline_mean_RMSE"] = baseline_rmse
                comparison["RMSE_improvement_vs_baseline_pct"] = (
                    (baseline_rmse - comparison["mean_validation_RMSE"])
                    / baseline_rmse * 100
                )

        comparison = comparison.sort_values(
            ["mean_validation_RMSE", "mean_validation_MAE", "validation_RMSE_std"]
        ).reset_index(drop=True)
        display(comparison.round(4))
else:
    print("Comparison is pending complete Phase 8 results.")

,model,mean_validation_MAE,mean_validation_RMSE,mean_validation_R2,validation_RMSE_std,mean_train_RMSE,RMSE_train_to_validation_gap,baseline_mean_RMSE,RMSE_improvement_vs_baseline_pct
0,Random Forest,10.6524,22.1057,0.9231,2.4354,11.4512,10.6545,18.972,-16.5179
1,HistGradientBoosting,10.6685,22.2287,0.9222,2.2381,15.9860,6.2427,18.972,-17.1662
2,Decision Tree,13.7251,26.3039,0.8911,2.3386,13.1238,13.1801,18.972,-38.6462
3,Linear Regression,13.2784,26.3649,0.8906,2.0278,23.7472,2.6177,18.972,-38.9675


## 3. Record the validation-based candidate model

In [3]:
if comparison is not None and not comparison.empty:
    selected_model = str(comparison.iloc[0]["model"])
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

    selection = {
        "selected_model": selected_model,
        "selection_metric": "mean_validation_RMSE",
        "tie_breakers": ["mean_validation_MAE", "validation_RMSE_std"],
        "final_test_used": False,
        "required_models_present": sorted(REQUIRED_MODELS),
    }
    PHASE9_SELECTION.write_text(json.dumps(selection, indent=2), encoding="utf-8")
    comparison.to_csv(PHASE9_COMPARISON, index=False)

    print("Candidate selected using development-period validation only:", selected_model)
    print("Created:", PHASE9_COMPARISON)
    print("Created:", PHASE9_SELECTION)
else:
    print("No candidate model is selected until all four Phase 8 models have results.")

Candidate selected using development-period validation only: Random Forest
Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase9_model_comparison.csv
Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase9_selected_model.json


## 4. Interpretation checklist

Check **validation RMSE**, **validation MAE**, **validation R²**, fold-to-fold stability, the train-to-validation RMSE gap, and improvement relative to the persistence baseline. The 2017 final holdout is intentionally excluded from this decision.